In [ ]:
!pip install patool==1.12
import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt
from google.colab import drive
import patoolib
import os
import shutil

# the leapgestrecog dataset is stored on google drive
drive.mount('/content/drive')
#patoolib.extract_archive("", outdir="/content/extracted_images") # upload dataset to google drive and paste the path here
# regroup the subject/gesture tree into one folder per gesture so the labels are gestures, not subjects
def make_gesture_dir(src_dir, dst_dir):
    for subject in sorted(os.listdir(src_dir)):
        subject_dir = os.path.join(src_dir, subject)
        if not os.path.isdir(subject_dir):
            continue
        for gesture in sorted(os.listdir(subject_dir)):
            gesture_dir = os.path.join(subject_dir, gesture)
            if not os.path.isdir(gesture_dir):
                continue
            os.makedirs(os.path.join(dst_dir, gesture), exist_ok=True)
            for img_name in os.listdir(gesture_dir):
                src = os.path.join(gesture_dir, img_name)
                dst = os.path.join(dst_dir, gesture, f'{subject}_{img_name}')
                if os.path.exists(dst):
                    continue
                # hard links avoid a second copy of the dataset on disk
                try:
                    os.link(src, dst)
                except OSError:
                    shutil.copy(src, dst)
    return dst_dir

data_dir = make_gesture_dir('/content/extracted_images/leapGestRecog', '/content/gesture_images')

batch_size = 32

# 224x224 is the input size vgg16 was trained on, the shared seed keeps the 70/30 split disjoint
train_dataset_raw = tf.keras.preprocessing.image_dataset_from_directory(
    data_dir,
    validation_split=0.3,
    subset="training",
    seed=123,
    image_size=(224, 224),
    batch_size=batch_size,
    label_mode='categorical'
)

# read class names here, the dataset returned by map() no longer has them
class_names = train_dataset_raw.class_names
num_classes = len(class_names)
print(f"Class Names: {class_names}")
print(f"Number of classes: {num_classes}")

# augmentation to reduce overfitting, applied to the training split only
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2)
])

# stop when validation loss stops improving and roll back to the best epoch
early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# the imagenet weights expect vgg16's own preprocessing, so it runs after augmentation
train_dataset = train_dataset_raw.map(lambda x, y: (preprocess_input(data_augmentation(x)), y))

validation_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    data_dir,
    validation_split=0.3,
    subset="validation",
    seed=123,
    image_size=(224, 224),
    batch_size=batch_size,
    label_mode='categorical'
)
validation_dataset = validation_dataset.map(lambda x, y: (preprocess_input(x), y))

# imagenet vgg16 as a frozen feature extractor, only the new head is trained
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

# new classification head, dropout 0.5 keeps the large dense layer from overfitting
model = models.Sequential([
    layers.Input(shape=(224, 224, 3)),
    base_model,
    layers.BatchNormalization(),
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(num_classes, activation='softmax')
])

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=20,
    callbacks=[early_stopping],
    verbose=1
)

val_loss, val_acc = model.evaluate(validation_dataset, verbose=2)
print(f"Validation accuracy: {val_acc:.4f}")

# accuracy and loss share one plot, the loss curves are dashed
plt.figure(figsize=(10, 6))

plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')

plt.plot(history.history['loss'], label='Training Loss', linestyle='--')
plt.plot(history.history['val_loss'], label='Validation Loss', linestyle='--')

plt.title('Model Accuracy and Loss')
plt.xlabel('Epoch')
plt.ylabel('Accuracy / Loss')

plt.legend(loc='lower right')

plt.show()
